# Virus-Wetter-Datenanalyse


---
## Block 1 — Setup & Daten laden


In [ ]:
from datetime import datetime
import os
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings
from skbio.diversity.alpha import shannon
from skbio.diversity import beta_diversity
from skbio.stats.ordination import pcoa
from scipy.stats import pearsonr
from scipy.stats import zscore


warnings.filterwarnings('ignore')

# Funktion, um den Inhalt eines Ordners zu löschen
def leere_den_ordner(ordner):
    if not os.path.exists(ordner):
        return

    for root, dirs, dateien in os.walk(ordner, topdown=False):
        for name in dateien:
            os.remove(os.path.join(root, name))
        for name in dirs:
            os.rmdir(os.path.join(root, name))

# Ausgabeordner
output_ordner = Path("../data/klima_analyse")
    
## Erstelle den Ordner, falls er noch nicht existiert
os.makedirs(output_ordner, exist_ok=True)
    
## Falls bereits Dtaen im Ordner vorhanden sind, lösche diese
leere_den_ordner(output_ordner)

staedte_nach_klima = {
    'Gemäßigt':    ['Copenhagen', 'Regina', 'Seattle', 'Melbourne'],
    'Subtropisch': ['Guangzhou'],
    'Tropisch':    ['Kuala Lumpur', 'Quito', 'Yaounde']
}

wetter_spalten = {
    'temperature_2m_mean (°C)':      'Temp',
    'rain_sum (mm)':                 'Regen',
    'relative_humidity_2m_mean (%)': 'Luftfeuchtigkeit',
}

ordner_fuer_staedte = "../data/samplings/"

staedte = {
    'Copenhagen':   ('Copenhagen_merged_reads.csv',  'Copenhagen_weather.csv'),
    'Guangzhou':    ('Guangzhou_merged_reads.csv',   'Guangzhou_weather.csv'),
    'Kuala Lumpur': ('KualaLumpur_merged_reads.csv', 'KualaLumpur_weather.csv'),
    'Melbourne':    ('Melbourne_merged_reads.csv',   'Melbourne_weather.csv'),
    'Quito':        ('Quito_merged_reads.csv',       'Quito_weather.csv'),
    'Regina':       ('Regina_merged_reads.csv',      'Regina_weather.csv'),
    'Seattle':      ('Seattle_merged_reads.csv',     'Seattle_weather.csv'),
    'Yaounde':      ('Yaounde_merged_reads.csv',     'Yaounde_weather.csv'),
}

staedte = {
    stadt: (
        os.path.join(ordner_fuer_staedte, reads),
        os.path.join(ordner_fuer_staedte, weather)
    )
    for stadt, (reads, weather) in staedte.items()
}


print("Lade ENA-Metadaten...")
ena_data = pd.read_csv('ena_data.tsv', sep='\t')
print(f"ENA-Daten: {len(ena_data)} Einträge\n")


alle_daten = {}
alle_virus  = {}
alle_pcoa   = {}

def lade_stadt(stadt, virus_datei, wetter_datei, ena_data, wetter_spalten):
    # Virus-Tabelle laden + aufbereiten 
    virus_df = pd.read_csv(virus_datei, index_col=0).fillna(0).T
    virus_df.columns = virus_df.columns.str.strip()
    virus_df = virus_df.drop('taxid', errors='ignore')

    # Shannon-Index pro Sample 
    shannon_werte = []
    for sample_id in virus_df.index:
        counts = virus_df.loc[sample_id].values
        s = shannon(counts, base=None, exp=False)
        shannon_werte.append({'run_accession': sample_id, 'shannon_index': s})
    alpha_div = pd.DataFrame(shannon_werte)

    # collection_date aus ena_data holen
    ena_city = (
        ena_data[ena_data['run_accession'].isin(alpha_div['run_accession'])]
        [['run_accession', 'collection_date']]
        .copy()
    )
    ena_city['collection_date'] = pd.to_datetime(ena_city['collection_date'])
    data = alpha_div.merge(ena_city, on='run_accession')

    # Wetterdaten laden + 5-Tage-Mittelwert berechnen
    weather_df = pd.read_csv(wetter_datei)
    weather_df['time'] = pd.to_datetime(weather_df['time'])

    for csv_spalte, col_name in wetter_spalten.items():
        data[col_name] = np.nan
        for i, row in data.iterrows():
            start = row['collection_date'] - pd.Timedelta(days=4)
            maske = (weather_df['time'] >= start) & (weather_df['time'] <= row['collection_date'])
            data.at[i, col_name] = weather_df.loc[maske, csv_spalte].mean()

    # NaN-Zeilen droppen -> data_clean
    data_clean = data.dropna(subset=list(wetter_spalten.values()) + ['shannon_index'])

    # Beta-Diversität + PCoA berechnen
    bc_dm = beta_diversity('braycurtis', virus_df.values, virus_df.index)
    pcoa_res = pcoa(bc_dm)

    return data_clean, virus_df, pcoa_res

for stadt, (virus_datei, wetter_datei) in staedte.items():
    data_clean, virus_df, pcoa_res = lade_stadt(stadt, virus_datei, wetter_datei, ena_data, wetter_spalten)
    alle_daten[stadt] = data_clean
    alle_virus[stadt] = virus_df
    alle_pcoa[stadt] = pcoa_res



Lade ENA-Metadaten...
ENA-Daten: 678 Einträge



---
## Block 1b — Explorative Datenanalyse (EDA)


In [22]:
for stadt, df in alle_daten.items():
    print(stadt)
    print(df.shape)
    df.info()
    print(df.isnull().sum())

Copenhagen
(12, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   run_accession     12 non-null     object        
 1   shannon_index     12 non-null     float64       
 2   collection_date   12 non-null     datetime64[ns]
 3   Temp              12 non-null     float64       
 4   Regen             12 non-null     float64       
 5   Luftfeuchtigkeit  12 non-null     float64       
dtypes: datetime64[ns](1), float64(4), object(1)
memory usage: 708.0+ bytes
run_accession       0
shannon_index       0
collection_date     0
Temp                0
Regen               0
Luftfeuchtigkeit    0
dtype: int64
Guangzhou
(14, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----        

---
## Block 2 — Deskriptive Analyse


In [23]:
df2 = pd.concat([alle_daten[stadt] for stadt in alle_daten])
wetter_min_max = {
     'Temp': (df2['Temp'].min(), df2['Temp'].max()),
     'Regen': (df2['Regen'].min(), df2['Regen'].max()),
     'Luftfeuchtigkeit': (df2['Luftfeuchtigkeit'].min(), df2['Luftfeuchtigkeit'].max())
}

wetter_norm = {
    'Temp': mcolors.Normalize(vmin=wetter_min_max['Temp'][0], vmax=wetter_min_max['Temp'][1]),
    'Regen': mcolors.Normalize(vmin= wetter_min_max['Regen'][0], vmax=wetter_min_max['Regen'][1]),
    'Luftfeuchtigkeit': mcolors.Normalize(vmin= wetter_min_max['Luftfeuchtigkeit'][0], vmax=wetter_min_max['Luftfeuchtigkeit'][1])
}
wetter_cmap = {
    'Temp': plt.cm.coolwarm,
    'Regen': plt.cm.Blues,
    'Luftfeuchtigkeit': plt.cm.BrBG
}

viren_pro_stadt = {}

def zeichne_heatmap(stadt, virus_df, wetter_min_max, wetter_norm, wetter_cmap, daten):
    # Proben Datum sortieren + Virus-Tabelle 
    daten_sortiert = daten.sort_values('collection_date')
    reihenfolge = daten_sortiert['run_accession']
    virus_sortiert = virus_df.loc[reihenfolge]

    # Seltene Viren rausfiltern (mind. in 25% der Proben)
    anzahl_proben = len(virus_sortiert)
    vorkommen = (virus_sortiert > 0).sum()
    schwelle = anzahl_proben * 0.25
    viren_behalten = vorkommen[vorkommen >= schwelle].index
    virus_gefiltert = virus_sortiert[viren_behalten]

    # Top-45 Viren nach Standardabweichung auswählen
    std_werte = virus_gefiltert.std()
    top_45 = std_werte.sort_values(ascending=False).head(45).index
    virus_top45 = virus_gefiltert[top_45]
    viren_pro_stadt[stadt] = set(top_45)

    # Werte pro Virus normalisieren
    virus_norm = virus_top45 / virus_top45.max()

   
    farben_temp = wetter_cmap['Temp'](wetter_norm['Temp'](daten_sortiert['Temp']))
    farben_regen = wetter_cmap['Regen'](wetter_norm['Regen'](daten_sortiert['Regen']))
    farben_luft = wetter_cmap['Luftfeuchtigkeit'](wetter_norm['Luftfeuchtigkeit'](daten_sortiert['Luftfeuchtigkeit']))

    col_colors = pd.DataFrame({
        'Temp': list(farben_temp),
        'Regen': list(farben_regen),
        'Luftfeuchtigkeit': list(farben_luft)
    }, index=daten_sortiert['run_accession'])


    xticklabels = [
        f"{datum.strftime('%Y-%m-%d')} | T:{temp:.1f} R:{regen:.1f}"
        for datum, temp, regen in zip(
            daten_sortiert['collection_date'],
            daten_sortiert['Temp'],
            daten_sortiert['Regen']
        )
    ]

    # Clustermap zeichnen 
    g = sns.clustermap(
        virus_norm.T,
        col_cluster=False,
        col_colors=col_colors,
        cmap='viridis',
        xticklabels=xticklabels
    )
    g.figure.suptitle(f'Virus-Heatmap {stadt}')
    g.savefig(f'{output_ordner}/heatmap_{stadt}.png')
    plt.close(g.figure)

# Für jede Stadt Heatmap zeichnen
for klimazone, staedte_liste in staedte_nach_klima.items():
    for stadt in staedte_liste:
        zeichne_heatmap(stadt, alle_virus[stadt], wetter_min_max, wetter_norm, wetter_cmap, alle_daten[stadt])
        
# Pro Klimazone gemeinsamen Viren berechnen
for klimazone, staedte_liste in staedte_nach_klima.items():
    if klimazone in ['Gemäßigt', 'Tropisch']: 
        viren_sets = [viren_pro_stadt[stadt] for stadt in staedte_liste]
        gemeinsame_viren = set.intersection(*viren_sets)
        print(f'{klimazone}: {gemeinsame_viren}')

Gemäßigt: {'Gemycircularvirus', 'Burzaovirus', 'Gemykibivirus', 'Longwangvirus', 'Kingevirus', 'Carjivirus', 'Kagunavirus', 'Burrovirus', 'Tobamovirus'}
Tropisch: {'Kahnovirus', 'Mamastrovirus', 'Burzaovirus', 'Gemykibivirus', 'Carjivirus', 'Fohxhuevirus', 'Cyclovirus', 'Aurodevirus', 'Kagunavirus', 'Afonbuvirus', 'Tobamovirus'}


---
## Block 3a — PCoA + Pearson Korrelation


---
## Block 3b — Zeitverlauf


---
## Block 4 — Saisonalität (Gemäßigt)


In [24]:
def monat_zu_Jahreszeit_Nordhalbkugel(monat):
    if monat in [3,4,5]:
        return 'Frühling'
    if monat in [6,7,8]:
        return 'Sommer'
    if monat in [9,10,11]:
        return 'Herbst'
    else:
        return 'Winter'

def monat_zu_Jahreszeit_Suedhalbkugel(monat):
    if monat in [3,4,5]:
        return 'Herbst'
    if monat in [6,7,8]:
        return 'Winter'
    if monat in [9,10,11]:
        return 'Frühling'
    else:
        return 'Sommer'

gemaessigte_staedte = staedte_nach_klima['Gemäßigt']

for stadt in gemaessigte_staedte:
    if stadt == 'Melbourne':
        alle_daten[stadt]['Saison'] = alle_daten[stadt]['collection_date'].dt.month.apply(monat_zu_Jahreszeit_Suedhalbkugel)
    else:
        alle_daten[stadt]['Saison'] = alle_daten[stadt]['collection_date'].dt.month.apply(monat_zu_Jahreszeit_Nordhalbkugel)

from scipy.stats import kruskal

ergebnisse = []

for stadt in gemaessigte_staedte:
    daten = alle_daten[stadt]

    # Shannon-Werte pro Saison gruppieren
    gruppen = [werte['shannon_index'].values for saison, werte in daten.groupby('Saison')]

    # Kruskal-Wallis-Test 
    test_statistik, p_wert = kruskal(*gruppen)

    # Saison mit höchstem Median-Shannon-Index 
    medianwerte = daten.groupby('Saison')['shannon_index'].median()
    peak_saison = medianwerte.idxmax()

   
    ergebnisse.append({
        'Stadt': stadt,
        'Test_Statistik': test_statistik,
        'P_Wert': p_wert,
        'Saison_Typ': 'Südhalbkugel' if stadt == 'Melbourne' else 'Nordhalbkugel',
        'Peak_Saison': peak_saison
    })

saisonalitaet_df = pd.DataFrame(ergebnisse)
saisonalitaet_df

,Stadt,Test_Statistik,P_Wert,Saison_Typ,Peak_Saison
0,Copenhagen,7.615385,0.022199,Nordhalbkugel,Frühling
1,Regina,6.262500,0.099515,Nordhalbkugel,Winter
2,Seattle,4.055833,0.255498,Nordhalbkugel,Winter
3,Melbourne,4.950000,0.175495,Südhalbkugel,Winter


---
## Block 5 — Regen- vs. Trockenzeit (Tropisch + Subtropisch)


In [25]:
from scipy.stats import mannwhitneyu

tropische_staedte = staedte_nach_klima['Subtropisch'] + staedte_nach_klima['Tropisch']

regenzeit_ergebnisse = []

for stadt in tropische_staedte:
    daten = alle_daten[stadt]

    # Median-Niederschlag Stadt als Schwelle 
    median_regen = daten['Regen'].median()

    # Proben in Regenzeit/Trockenzeit einteilen
    daten['Periode'] = daten['Regen'].apply(lambda r: 'Regenzeit' if r >= median_regen else 'Trockenzeit')

    # Shannon-Werte für beide Gruppen trennen
    regenzeit_werte = daten[daten['Periode'] == 'Regenzeit']['shannon_index']
    trockenzeit_werte = daten[daten['Periode'] == 'Trockenzeit']['shannon_index']

    # Mann-Whitney-U-Test
    test_statistik, p_wert = mannwhitneyu(regenzeit_werte, trockenzeit_werte)

    regenzeit_ergebnisse.append({
        'Stadt': stadt,
        'Test_Statistik': test_statistik,
        'P_Wert': p_wert,
        'Median_Regen': median_regen
    })

regenzeit_df = pd.DataFrame(regenzeit_ergebnisse)
regenzeit_df

,Stadt,Test_Statistik,P_Wert,Median_Regen
0,Guangzhou,14.0,0.208625,3.46
1,Kuala Lumpur,24.0,0.694328,11.44
2,Quito,14.0,0.120591,1.82
3,Yaounde,30.0,0.878477,4.47


---
## Block 6 — Pearson-Korrelation


In [26]:
from scipy.stats import pearsonr

regression_ergebnisse = []

for stadt, daten in alle_daten.items():
    for variable in ['Temp', 'Regen', 'Luftfeuchtigkeit']:
        r, p_wert = pearsonr(daten['shannon_index'], daten[variable])
        regression_ergebnisse.append({
            'Stadt': stadt,
            'Variable': variable,
            'r': r,
            'P_Wert': p_wert
        })

regression_df = pd.DataFrame(regression_ergebnisse)
regression_df

,Stadt,Variable,r,P_Wert
0,Copenhagen,Temp,-0.442282,0.149953
1,Copenhagen,Regen,0.047295,0.883957
2,Copenhagen,Luftfeuchtigkeit,-0.237954,0.456428
3,Guangzhou,Temp,-0.727926,0.003162
4,Guangzhou,Regen,-0.505173,0.065389
5,Guangzhou,Luftfeuchtigkeit,-0.676583,0.007879
6,Kuala Lumpur,Temp,-0.024632,0.930565
7,Kuala Lumpur,Regen,-0.152755,0.586786
8,Kuala Lumpur,Luftfeuchtigkeit,-0.256252,0.356589
9,Melbourne,Temp,-0.680393,0.005246


---
## Block 7 — Ausreißer-Erkennung (Z-Score)


In [27]:
from scipy.stats import zscore

anomalie_eintraege = []
schwelle_z = 2

# Shannon-Index-Ausreißer pro Stadt
for stadt, daten in alle_daten.items():
    z_werte = zscore(daten['shannon_index'])
    for run_acc, z in zip(daten['run_accession'], z_werte):
        if abs(z) > schwelle_z:
            anomalie_eintraege.append({
                'Stadt': stadt,
                'run_accession': run_acc,
                'Typ': 'Shannon_Index',
                'Z_Score': z,
                'Bewertung': 'Auffällig'
            })

# Virus-Häufigkeits-Ausreißer pro Stadt und Virus
for stadt, virus_df in alle_virus.items():
    for virus in virus_df.columns:
        werte = virus_df[virus]
        if werte.std() == 0:
            continue  
        z_werte = zscore(werte)
        for run_acc, z in zip(virus_df.index, z_werte):
            if abs(z) > schwelle_z:
                anomalie_eintraege.append({
                    'Stadt': stadt,
                    'run_accession': run_acc,
                    'Typ': f'Virus_{virus}',
                    'Z_Score': z,
                    'Bewertung': 'Auffällig'
                })


anomalie_df = pd.DataFrame(anomalie_eintraege)


anomalie_df = anomalie_df.reindex(anomalie_df['Z_Score'].abs().sort_values(ascending=False).index)

# Wetter-Z-Scores pro Stadt berechnen
wetter_z = {}
for stadt, daten in alle_daten.items():
    wetter_z[stadt] = pd.DataFrame({
        'run_accession': daten['run_accession'],
        'Temp_z': zscore(daten['Temp']),
        'Regen_z': zscore(daten['Regen']),
        'Luftfeuchtigkeit_z': zscore(daten['Luftfeuchtigkeit'])
    })

# Hat auffälige Probe -> auffälliges Wetter
def pruefe_wetterbezug(stadt, run_acc, schwelle=1.5):
    zeile = wetter_z[stadt][wetter_z[stadt]['run_accession'] == run_acc]
    if zeile.empty:
        return 'unerklärt'
    werte = zeile[['Temp_z', 'Regen_z', 'Luftfeuchtigkeit_z']].abs().values[0]
    return 'möglicherweise wetterbedingt' if (werte > schwelle).any() else 'unerklärt'

anomalie_df['Bewertung'] = anomalie_df.apply(
    lambda zeile: pruefe_wetterbezug(zeile['Stadt'], zeile['run_accession']), axis=1
)

anomalie_df

,Stadt,run_accession,Typ,Z_Score,Bewertung
1759,Yaounde,ERR14789286,Virus_Husavirus,3.872983,unerklärt
1969,Yaounde,ERR14789286,Virus_Moazamivirus,3.872983,unerklärt
1965,Yaounde,ERR14789292,Virus_Ferozepurvirus,3.872983,möglicherweise wetterbedingt
1671,Yaounde,ERR14789292,Virus_Uetakevirus,3.872983,möglicherweise wetterbedingt
1678,Yaounde,ERR14789282,Virus_Betacoronavirus,3.872983,möglicherweise wetterbedingt
...,...,...,...,...,...
498,Kuala Lumpur,ERR14789310,Virus_Kahnovirus,-2.010638,unerklärt
1342,Regina,ERR14789363,Virus_Burzaovirus,2.010306,möglicherweise wetterbedingt
653,Melbourne,ERR14789264,Virus_Przondovirus,2.006163,unerklärt
524,Kuala Lumpur,ERR14789311,Virus_Diorhovirus,2.004073,möglicherweise wetterbedingt


---
## Block 8 — Export für Datenbank


In [28]:
# shannon.csv exportieren
shannon_export = pd.concat([
    alle_daten[stadt][['run_accession', 'shannon_index']]
    for stadt in alle_daten
])
shannon_export.to_csv(f'{ordner}/shannon.csv', index=False)

# regression.csv exportieren
regression_df.to_csv(f'{ordner}/regression.csv', index=False)

# seasonality.csv exportieren
saisonalitaet_df.to_csv(f'{ordner}/seasonality.csv', index=False)

# anomalies.csv exportieren
anomalie_df.to_csv(f'{ordner}/anomalies.csv', index=False)

# seasonal_viruses.csv exportieren
seasonal_viren_eintraege = []
for klimazone, staedte_liste in städte_nach_klima.items():
    if klimazone in ['Gemäßigt', 'Tropisch']:
        viren_sets = [viren_pro_stadt[stadt] for stadt in staedte_liste]
        gemeinsame_viren = set.intersection(*viren_sets)
        for virus in gemeinsame_viren:
            seasonal_viren_eintraege.append({'Klimazone': klimazone, 'Virus': virus})

seasonal_viruses_df = pd.DataFrame(seasonal_viren_eintraege)
seasonal_viruses_df.to_csv(f'{ordner}/seasonal_viruses.csv', index=False)

OSError: Cannot save file into a non-existent directory: 'Datenanalyse_2026-06-21_21-02'

---
## Block 9 — Limitierungen & Ausblick

### Limitierungen

**Stichprobengröße:** n=12–16 Proben pro Stadt. Statistische Aussagen sind mit Vorsicht zu interpretieren.



## Block 9.1 - Quellen 

### Block 1
https://scikit.bio/docs/dev/generated/skbio.diversity.alpha.shannon.html
https://scikit.bio/docs/latest/diversity.html 
https://scikit.bio/docs/dev/generated/skbio.stats.ordination.pcoa.html 

### Block 2
https://seaborn.pydata.org/generated/seaborn.clustermap.html

### Block 3a
### Block 3b

### Block 4
https://www.geeksforgeeks.org/python/how-to-perform-a-kruskal-wallis-test-in-python/

### Block 5
https://www.geeksforgeeks.org/machine-learning/mann-whitney-u-test-2/

### Block 6
https://www.geeksforgeeks.org/data-science/pearson-correlation-in-data-science/

### Block 7
https://www.geeksforgeeks.org/python/scipy-stats-zscore-function-python/
https://medium.com/@whyamit101/understanding-z-score-with-numpy-bc8b23f81639

### Block 8



